# 🚀 IndexTTS-2.5 · 3 步部署 (Colab)

运行前: `运行时 → 更改运行时类型 → 硬件加速器 = GPU` (T4 即可)。

**步骤 1** 一键安装 → **步骤 2** 后台启动服务(API+WebUI, 自动打印公网 URL) → **步骤 3** 查看实时日志。

> 仓库地址: `https://github.com/infinite-gaming-studio/index-tts`（本仓库）
>
> 下载慢可先运行 `%env HF_ENDPOINT=https://hf-mirror.com`；也可改用 ModelScope: 在步骤1前运行 `%env MODEL_SOURCE=modelscope`
>
> 步骤 2 默认启动 **API + WebUI 双服务**（`SERVICE=both`）：API 在 `{公网URL}/api/tts`（文档 `/docs`），WebUI 在 `{公网URL}/`
>
> 隧道默认 **ngrok**：运行步骤 2 时会弹出输入框粘贴 authtoken（免费注册 https://dashboard.ngrok.com）；免注册可用 `%env TUNNEL=cf`


In [ ]:
# ===== 步骤 1/3: 一键部署（克隆本仓库 → 装依赖 → 下载 IndexTTS-2.5 权重）=====
REPO_URL = "https://github.com/infinite-gaming-studio/index-tts.git"

!git clone --depth 1 $REPO_URL index-tts
%cd index-tts
!bash deploy/scripts/setup.sh

In [ ]:
# ===== 步骤 2/3: 后台启动服务 (API + WebUI) + 公网隧道 (实时输出日志) =====
# 默认 SERVICE=both (API 端口 8000 + WebUI 挂载 /ui), 可用 %env SERVICE=api|webui 切换
# 默认 TUNNEL=ngrok (需 %env NGROK_TOKEN=你的authtoken); 想用 cloudflare 免注册: %env TUNNEL=cf
# 本单元格实时打印部署日志; 公网URL/API Key/错误等会带标记突出显示; 完整日志: api.log
import os, re, time

os.environ["SERVICE"] = "both"  # api | webui | both
# 隧道默认 ngrok; 未设置 NGROK_TOKEN 时下方会弹出输入框让你粘贴 authtoken
if os.environ.get("TUNNEL", "ngrok") == "ngrok" and not os.environ.get("NGROK_TOKEN"):
    print("🔐 ngrok 需要 authtoken, 免费获取: https://dashboard.ngrok.com (Dashboard → Your Authtoken)")
    from getpass import getpass
    tok = getpass("粘贴你的 ngrok authtoken 后回车 (输入时不显示): ").strip()
    if tok:
        os.environ["NGROK_TOKEN"] = tok
    else:
        print("❌ 未输入 NGROK_TOKEN, 隧道无法启动。")
        print("   可改用免注册隧道: %env TUNNEL=cf 后重跑本单元格")
        raise SystemExit(1)

LOG_FILE = "serve_console.log"
if os.path.exists(LOG_FILE):
    os.remove(LOG_FILE)  # 清掉上次的日志, 避免读到旧内容
os.system(f"nohup bash deploy/scripts/serve.sh > {LOG_FILE} 2>&1 &")

# ---- 重要日志识别规则: 命中即突出显示 ----
url_pattern = re.compile(r"https://[a-z0-9-]+\.(?:ngrok-free\.app|ngrok\.app|ngrok\.io|trycloudflare\.com)")
key_pattern = re.compile(r"API Key: (\S+)")
error_markers = ("!! ", "ERR_NGROK", "Traceback", "failed to connect", "FATAL")
BANNER = "=" * 54

def tag(line):
    """给重要日志加标记前缀"""
    if "API Key:" in line:
        return "🔑 " + line
    if url_pattern.search(line) or "Public URL" in line:
        return "🌐 " + line
    if "服务已就绪" in line:
        return "🟢 " + line
    if any(mk in line for mk in error_markers):
        return "❌ " + line
    if line.startswith("==>"):
        return "▶ " + line
    return line

found_url, found_key = None, None
seen = 0
for i in range(120):  # 最多约 10 分钟
    time.sleep(5)
    try:
        lines = open(LOG_FILE, encoding="utf-8", errors="ignore").read().splitlines()
    except FileNotFoundError:
        continue
    for ln in lines[seen:]:
        ln = ln.strip()
        if ln:
            print(tag(ln))
        m = url_pattern.search(ln)
        if m and not found_url:
            found_url = m.group(0)
            print(f"\n{BANNER}\n✅ 公网地址: {found_url}\n{BANNER}")
        m = key_pattern.search(ln)
        if m and not found_key:
            found_key = m.group(1)
            print(f"\n{BANNER}\n🔑 API Key: {found_key}\n{BANNER}")
    seen = len(lines)
    if found_url:
        break
    if any(mk in ln for ln in lines for mk in error_markers):
        break  # 启动出错立即停止, 下方直接输出原因

print()
if found_url:
    print("✅ 部署完成!")
    print(f"   公网地址: {found_url}")
    if found_key:
        print(f"   API Key:  {found_key}")
        print(f"   调用示例: curl -H \"Authorization: Bearer {found_key}\" -F text='你好' -F spk_audio=@ref.wav {found_url}/api/tts")
    print(f"   API 文档: {found_url}/docs  ·  WebUI: {found_url}/  ·  健康检查: {found_url}/api/health")
else:
    print("⚠️ 未获取到公网地址 (可能启动失败), 最近日志:")
    os.system("tail -40 " + LOG_FILE)

In [ ]:
# ===== 步骤 3/3: 查看实时日志 (排查问题) =====
# 中断本单元格仅停止查看日志, 不影响后台服务。
# 默认 SERVICE=both, 日志在 api.log (WebUI 模式才用 webui.log)
!bash deploy/scripts/logs.sh api -f

In [ ]:
# ===== 防 Colab 空闲断连: 定时点击"连接"按钮 (需保持本标签页打开) =====
# Colab 免费版约 90 分钟无操作会断连; 此脚本每 60s 自动点一次连接按钮保活。
# 停止: 中断本单元格; 重新运行即可恢复。
from IPython.display import display, Javascript

display(Javascript("""
function clickConnect(){
  const btn = document.querySelector('colab-connect-button');
  if (btn) btn.click();
}
setInterval(clickConnect, 60000);
"""))
print("✅ 防断连脚本已启动 (每 60s 自动重连; 请保持浏览器标签页打开)")

## (可选) 合成验证

以下为**默认注释**的参考代码，如需使用请取消注释后运行。

In [ ]:
# ===== (可选) 初始化 + 语音克隆合成 =====
# from indextts.utils.examples_downloader import ensure_examples_available
# ensure_examples_available()
#
# from indextts.infer_v2_5 import IndexTTS2
# tts = IndexTTS2(cfg_path="checkpoints/config.yaml", model_dir="checkpoints", use_bf16=True)
#
# tts.infer(
#     spk_audio_prompt="examples/voice_01.wav",
#     text="你好，我是 IndexTTS-2.5，欢迎测试多语言语音合成。",
#     lang="ZH",
#     output_path="output_zh.wav",
#     verbose=True,
# )

In [ ]:
# ===== (可选) 播放 + 下载结果 =====
# from IPython.display import Audio
# from google.colab import files
#
# Audio("output_zh.wav")
# files.download("output_zh.wav")